# Pipeline tiền xử lý — WiFi Fingerprinting

Đồ án tốt nghiệp Nhóm 15 · Khoa CNTT · Đại học Đà Lạt

Notebook này **không chép lại logic xử lý**. Nó clone kho mã rồi gọi thẳng gói `ml/`
trong đó, nên chỉ có **một nguồn sự thật duy nhất**. Sửa mã ở kho là Colab chạy theo,
không bao giờ lệch nhau.

Dữ liệu thô đã nằm sẵn trong kho — **không phải upload lại mỗi phiên** như trước.

| Chạy gì | Ở đâu |
|---|---|
| Muốn kết quả ngay | Phần 1, chạy đúng hai ô |
| Muốn xem từng bước để chụp màn hình đưa vào báo cáo | Phần 2 |

---
## Phần 0 — Chuẩn bị

Chạy một lần mỗi phiên Colab. Mất khoảng 30 giây.

In [ ]:
# Clone kho mã (hoặc cập nhật nếu đã có sẵn từ lần chạy trước trong phiên này)
import os, subprocess, sys

REPO = "https://github.com/DoAnTotNghiep-Indoor/Indoor_Positoning_System-DATN.git"
DIR  = "Indoor_Positoning_System-DATN"

if os.path.isdir(DIR):
    subprocess.run(["git", "-C", DIR, "pull", "--quiet"], check=False)
    print("Đã cập nhật kho mã có sẵn.")
else:
    subprocess.run(["git", "clone", "--quiet", REPO, DIR], check=True)
    print("Đã clone kho mã.")

os.chdir(DIR)
sys.path.insert(0, os.getcwd())
print("Thư mục làm việc:", os.getcwd())

In [ ]:
# Colab đã có sẵn pandas, numpy, scikit-learn, joblib.
# Chỉ cài thêm nếu thiếu, tránh mất thời gian cài lại không cần thiết.
import importlib.util, subprocess, sys

thieu = [m for m in ("pandas", "numpy", "sklearn", "joblib")
         if importlib.util.find_spec(m) is None]

if thieu:
    ten_goi = {"sklearn": "scikit-learn"}
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    *[ten_goi.get(m, m) for m in thieu]], check=True)
    print("Đã cài:", thieu)
else:
    print("Đủ thư viện, không cần cài thêm.")

import pandas as pd, numpy as np, sklearn
print(f"pandas {pd.__version__} · numpy {np.__version__} · scikit-learn {sklearn.__version__}")

In [ ]:
# Kiểm tra dữ liệu đầu vào đã có đủ chưa
from ml import config

for ten, duong_dan in [("Dữ liệu thô", config.RAW_CSV),
                       ("Bảng toạ độ", config.REFERENCE_POINTS_CSV)]:
    if duong_dan.exists():
        print(f"OK   {ten:14s} {duong_dan.stat().st_size/1e6:6.2f} MB  {duong_dan.name}")
    else:
        print(f"THIẾU {ten:14s} {duong_dan}")

---
## Phần 1 — Chạy toàn bộ bằng một lệnh

Cách nhanh nhất. Chạy hết 12 bước và ghi đủ artifact.

In [ ]:
!python -m ml.pipeline

### Các biến thể

Bỏ dấu `#` ở dòng nào muốn chạy.

In [ ]:
# Thí nghiệm ngưỡng lọc AP — chạy cả ba để có bảng so sánh cho báo cáo
# !python -m ml.pipeline --min-appear-rate 0.0
# !python -m ml.pipeline --min-appear-rate 0.10

# Tách riêng một thiết bị làm tập test (cần dữ liệu từ >= 2 máy)
# !python -m ml.pipeline --split device_holdout

# Dừng lại nếu còn điểm tham chiếu chưa đo toạ độ
# !python -m ml.pipeline --strict-coords

# Lọc nhiễu trên toàn bộ dữ liệu như bản Colab cũ (để đối chứng, không khuyến nghị)
# !python -m ml.pipeline --hampel-all

---
## Phần 2 — Từng bước một

Phần này cho ra **cùng kết quả** với Phần 1, nhưng hiện rõ số liệu trung gian của
mỗi bước — tiện chụp màn hình đưa vào chương mô tả dữ liệu.

In [ ]:
import pandas as pd, numpy as np
from ml import config
from ml.preprocess import load, pivot, coords, filter as ap_filter, missing, denoise, split, scale

pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 120)

### Bước 1 — Nạp dữ liệu thô

Đọc và kiểm tra ngay tại đây. Dữ liệu sai định dạng phải dừng lập tức, đừng để lỗi
lan xuống tận bước chuẩn hoá rồi mới phát hiện.

In [ ]:
df = load.load_raw()
mo_ta = load.describe_raw(df)

for k, v in mo_ta.items():
    print(f"  {k:22s} {v}")

df.head()

### Bước 2 — Gom theo lần quét

Cột `Time` làm `scan_id`. Lý do: "WiFi fingerprint serial number" chỉ lặp 1..N theo
từng đợt, không duy nhất toàn cục.

**Sửa so với bản cũ:** cột `Orientation Azimuth (°)` ghi đơn vị là độ nhưng giá trị
thực nằm trong `[-pi, pi]` — tức **radian**. Hàm `build_scan_meta` tự đổi sang độ.

In [ ]:
df = pivot.build_scan_id(df)
scan_meta = pivot.build_scan_meta(df)

print("Số lần quét:", len(scan_meta))
print("Số thiết bị:", scan_meta['device_id'].nunique())
print(f"Hướng đặt máy sau khi đổi sang độ: "
      f"{scan_meta['azimuth_deg'].min():.1f}° → {scan_meta['azimuth_deg'].max():.1f}°")
print("\nSố lần quét theo từng điểm tham chiếu:")
print(scan_meta['rp_id'].value_counts().sort_index().head(10))

### Bước 3 — Pivot sang bảng vân tay

Mỗi dòng một lần quét, mỗi cột một BSSID.

**Sửa so với bản cũ:** thứ tự cột được sắp xếp tường minh bằng `sorted()` thay vì phó
mặc `pivot_table`. Thứ tự này chính là hợp đồng dữ liệu với backend nên không được để
nó phụ thuộc phiên bản pandas.

In [ ]:
fingerprint, ap_cols = pivot.to_wide(df, scan_meta)

print(f"Bảng vân tay: {fingerprint.shape[0]} lần quét × {len(ap_cols)} cột AP")
print("Thứ tự cột đã sắp xếp:", ap_cols == sorted(ap_cols))
fingerprint.iloc[:5, :10]

### Bước 4 — Ghép toạ độ thật (x, y)

**Bước quyết định cả đồ án.** Không có `(x, y)` thì chỉ phân lớp được điểm tham chiếu
chứ không hồi quy được toạ độ, tức mất luôn cải tiến chính so với đồ án cũ.

Toạ độ lấy từ `data/reference/reference_points.csv`, nguồn gốc là **Bảng 4 trang 46**
đồ án CTK45. RP41 là điểm nhóm 2025 tự thêm nên chưa có toạ độ — mặc định bỏ 20 mẫu
của nó, vẫn còn 97,5% dữ liệu để huấn luyện.

In [ ]:
rp = coords.load_reference_points()
print(f"Bảng toạ độ: {len(rp)} điểm, {rp['x'].notna().sum()} điểm đã có toạ độ")

fingerprint, tk_toa_do = coords.attach_coordinates(fingerprint)
for k, v in tk_toa_do.items():
    print(f"  {k:22s} {v}")

fingerprint[['scan_id', 'rp_id', 'x', 'y']].head()

### Bước 5 — Lọc AP hiếm gặp

Loại AP xuất hiện dưới 20% số lần quét — thường là điểm phát cá nhân hoặc AP ở toà nhà
khác, chỉ làm nhiễu mô hình.

Bảng tỉ lệ xuất hiện trả về kèm để vẽ biểu đồ biện minh cho ngưỡng đã chọn.

In [ ]:
fingerprint, ap_cols, ty_le = ap_filter.filter_access_points(fingerprint, ap_cols)

print(f"Giữ {len(ap_cols)}/{len(ty_le)} AP (ngưỡng ≥ {config.MIN_APPEAR_RATE:.0%})")
print("\n10 AP có tỉ lệ xuất hiện thấp nhất (bị loại):")
print((ty_le.head(10) * 100).round(1).astype(str) + '%')

In [ ]:
# Biểu đồ biện minh ngưỡng — lưu vào reports/figures/ để chèn vào báo cáo
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(range(len(ty_le)), ty_le.values * 100, width=1.0,
       color=["#c44" if v < config.MIN_APPEAR_RATE else "#0F6E8C" for v in ty_le.values])
ax.axhline(config.MIN_APPEAR_RATE * 100, ls="--", lw=1, color="#333")
ax.text(1, config.MIN_APPEAR_RATE * 100 + 2, f"ngưỡng {config.MIN_APPEAR_RATE:.0%}", fontsize=9)
ax.set_xlabel("Access Point (xếp theo tỉ lệ xuất hiện)")
ax.set_ylabel("Tỉ lệ xuất hiện (%)")
ax.set_title("Tỉ lệ xuất hiện của các AP — cơ sở chọn ngưỡng lọc")
fig.tight_layout()

(config.REPORTS_DIR / "figures").mkdir(parents=True, exist_ok=True)
duong_dan = config.REPORTS_DIR / "figures" / "ap_appearance_rate.png"
fig.savefig(duong_dan, dpi=150)
print("Đã lưu", duong_dan)
plt.show()

### Bước 6 — Loại mẫu quét quá nghèo

Ngưỡng tối thiểu 6 AP mỗi mẫu. Phải chạy **sau** bước 5: lọc AP xong mới biết mẫu nào
còn quá ít AP hợp lệ.

Với bộ dữ liệu hiện tại không mẫu nào bị loại — mẫu nghèo nhất vẫn bắt được 15/36 AP.
Đó là tín hiệu chất lượng thu thập tốt, không phải bước bị bỏ qua.

In [ ]:
fingerprint, tk_scan = ap_filter.filter_sparse_scans(fingerprint, ap_cols)
for k, v in tk_scan.items():
    print(f"  {k:24s} {v}")

### Bước 7 — Điền RSSI thiếu bằng hằng số động

Thay vì gán cố định −98, tính `min(RSSI toàn tập) − 1`. Cách động này tránh việc giá trị
"không bắt được" trùng với tín hiệu yếu thật.

Giá trị này **phải** được ghi vào `feature_list.json`: backend gặp BSSID không có trong
lần quét cũng phải điền đúng con số đó.

In [ ]:
fingerprint, gia_tri_thieu, so_o_trong = missing.fill_missing(fingerprint, ap_cols)

print(f"Giá trị điền thiếu : {gia_tri_thieu} dBm")
print(f"Số ô đã điền       : {so_o_trong:,}")
print(f"Còn ô trống nào?   : {fingerprint[ap_cols].isna().sum().sum()}")

### Bước 9 chạy TRƯỚC bước 8 — đây là thay đổi quan trọng nhất

Bản Colab cũ chạy Hampel (bước 8) rồi mới chia tập (bước 9). Cách đó **sai về phương
pháp**, vì hai lý do:

1. **Rò rỉ dữ liệu.** Hampel thay giá trị lệch bằng trung vị của nhóm cùng `rp_id`. Tính
   trước khi chia tập nghĩa là trung vị đó bao gồm cả mẫu test — tập test tự làm sạch
   chính nó. Cùng loại lỗi với việc fit scaler trên toàn bộ dữ liệu, thứ mà README của
   dự án đã cấm.

2. **Backend không làm được điều tương tự.** Lúc chạy thật, backend chỉ nhận **một** lần
   quét và **không biết** nó thuộc điểm tham chiếu nào, nên không thể lọc Hampel. Tập
   test đã lọc nhiễu là tập test dễ hơn thực tế — đúng kiểu lạc quan giả đã khiến đồ án
   cũ đạt 88–97% trong phòng lab nhưng rớt xuống 5–12 m ngoài thực tế.

Nên: chia tập trước, rồi chỉ lọc nhiễu trên tập train.

In [ ]:
fingerprint, tk_chia = split.split_dataset(fingerprint)

print("Chiến lược:", tk_chia['chien_luoc'])
print("Số mẫu    :", tk_chia['so_mau'])
print("Số điểm   :", tk_chia['so_rp_moi_tap'])
if 'canh_bao' in tk_chia:
    print("\nCẢNH BÁO:", tk_chia['canh_bao'])

### Bước 8 — Lọc nhiễu Hampel, chỉ trên tập train

**Sửa so với bản cũ:** vector hoá toàn bộ. Bản cũ lặp Python qua từng cột rồi gọi
`groupby.transform` 36 lần; ở đây tính một lần cho cả ma trận.

In [ ]:
la_train = fingerprint['split'] == 'train'
da_loc, so_ngoai_lai = denoise.hampel_filter(fingerprint.loc[la_train], ap_cols)
fingerprint = pd.concat([da_loc, fingerprint.loc[~la_train]], ignore_index=True)

print(f"Thay {so_ngoai_lai:,} giá trị ngoại lai — chỉ trên tập train")
print(f"Số dòng giữ nguyên: {len(fingerprint)}")

### Lưu bản CHƯA chuẩn hoá

Bước này không có trong bản cũ, và chính vì thiếu nó mà lần chạy Colab trước đã mất khả
năng quay về đơn vị dBm. Lưu trước khi scale.

In [ ]:
cot_meta = [c for c in config.META_COLS if c in fingerprint.columns]
fingerprint = fingerprint[cot_meta + ap_cols]

config.PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
duong_dan_raw = config.PROCESSED_DIR / "fingerprint_dataset_raw.csv"
fingerprint.sort_values("rp_id", kind="stable").to_csv(duong_dan_raw, index=False)

print("Đã lưu", duong_dan_raw.name)
print(f"RSSI vẫn ở đơn vị dBm: {fingerprint[ap_cols].to_numpy().min():.0f} "
      f"→ {fingerprint[ap_cols].to_numpy().max():.0f}")

### Bước 10 — Chuẩn hoá min-max, fit chỉ trên tập train

Giá trị của validation và test **được phép** nằm ngoài `[0, 1]`. Đó không phải lỗi mà là
bằng chứng scaler chưa từng nhìn thấy hai tập đó.

In [ ]:
fingerprint, scaler, tk_scale = scale.scale_dataset(fingerprint, ap_cols)

print(f"Khoảng giá trị sau chuẩn hoá: {tk_scale['nho_nhat']:.3f} → {tk_scale['lon_nhat']:.3f}")
print("\nTheo từng tập:")
for ten in ('train', 'validation', 'test'):
    con = fingerprint.loc[fingerprint['split'] == ten, ap_cols].to_numpy()
    print(f"  {ten:11s} {con.min():7.3f} → {con.max():6.3f}")
print("\nTrain nằm gọn trong [0, 1]; val/test tràn ra ngoài -> scaler chưa thấy chúng.")

### Bước 11 + 12 — Ghi artifact và sắp xếp

`artifacts/` là hợp đồng dữ liệu với backend. Ba tệp trong đó phải **sinh ra từ cùng một
lần chạy**, không được trộn từ các lần khác nhau.

In [ ]:
import json, joblib
from datetime import datetime

fingerprint = fingerprint.sort_values("rp_id", kind="stable").reset_index(drop=True)

for thu_muc in (config.ARTIFACTS_DIR, config.SPLITS_DIR):
    thu_muc.mkdir(parents=True, exist_ok=True)

feature_list = {
    "ap_columns": ap_cols,
    "feature_count": len(ap_cols),
    "missing_rssi_value": gia_tri_thieu,
    "min_ap_per_scan": config.MIN_AP_PER_SCAN,
    "min_appear_rate": config.MIN_APPEAR_RATE,
    "target_columns": config.TARGET_COLS,
    "scaler_file": config.SCALER_PKL,
    "created_at": datetime.now().isoformat(timespec="seconds"),
}
(config.ARTIFACTS_DIR / config.FEATURE_LIST_JSON).write_text(
    json.dumps(feature_list, ensure_ascii=False, indent=2), encoding="utf-8")
joblib.dump(scaler, config.ARTIFACTS_DIR / config.SCALER_PKL)

fingerprint.to_csv(config.PROCESSED_DIR / "fingerprint_dataset_sorted.csv", index=False)
for ten in ("train", "validation", "test"):
    fingerprint[fingerprint['split'] == ten].to_csv(config.SPLITS_DIR / f"{ten}.csv", index=False)

print(f"Dataset cuối: {fingerprint.shape[0]} × {fingerprint.shape[1]}")
print(f"Số đặc trưng: {len(ap_cols)}")
print("\nĐã ghi:")
for p in sorted(config.ARTIFACTS_DIR.glob('*')) + sorted(config.SPLITS_DIR.glob('*.csv')):
    if p.is_file() and p.stat().st_size:
        print(f"  {p.stat().st_size/1024:8.1f} KB  {p.relative_to(config.ROOT_DIR)}")

---
## Phần 3 — Kiểm tra hợp đồng dữ liệu

Chạy bộ test thật của dự án. Bài quan trọng nhất là
`test_dao_thu_tu_van_ra_ket_qua_giong_het` — nó chặn đúng lỗi mà đồ án CTK45 mắc phải:
client gửi đủ số phần tử nhưng sai thứ tự AP thì mô hình vẫn chạy và trả kết quả sai
hoàn toàn, không một cảnh báo nào.

In [ ]:
!python -m pytest tests/ -q

---
## Phần 4 — Tải artifact về máy

Tải cả thư mục dưới dạng một file nén, thay vì tải từng file như bản cũ — tránh đúng
tình huống đã xảy ra lần trước là chỉ tải được file CSV còn `scaler.pkl` thì mất.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("ips_artifacts", "zip", ".", "artifacts")
shutil.make_archive("ips_data", "zip", ".", "data/splits")

for ten in ("ips_artifacts.zip", "ips_data.zip"):
    print(f"{os.path.getsize(ten)/1024:8.1f} KB  {ten}")
    files.download(ten)

### Hoặc đẩy thẳng lên GitHub

Cách này tốt hơn tải về máy: cả nhóm dùng chung một bản, không ai phải gửi file cho ai.

Cần **Personal Access Token** có quyền `repo` — tạo tại
`github.com/settings/tokens`. Đừng dán token thẳng vào ô mã rồi lưu notebook lên kho.

In [ ]:
# from getpass import getpass
# token = getpass("GitHub token: ")
#
# !git config user.name  "Ngocon2004"
# !git config user.email "buivhai2004@gmail.com"
# !git add artifacts/ data/splits/ data/processed/ reports/
# !git commit -q -m "Cập nhật artifact sau khi chạy lại pipeline trên Colab"
# !git push -q https://{token}@github.com/DoAnTotNghiep-Indoor/Indoor_Positoning_System-DATN.git main
# print("Đã đẩy lên GitHub.")

---

## Vì sao notebook này không chép lại logic

Bản cũ có 12 file `.py` rời, mỗi file tự chứa logic riêng. Vấn đề: sửa ở kho mã thì
Colab không đổi theo, và ngược lại — hai bản **lệch dần** mà không ai biết cho tới lúc
kết quả khác nhau.

Notebook này gọi thẳng gói `ml/` trong kho, nên:

- Sửa một chỗ, cả hai nơi cùng đổi.
- Bộ test chạy được trên cả máy lẫn Colab, kiểm chứng cùng một đoạn mã.
- `python -m ml.pipeline` cho ra kết quả **giống hệt** Phần 2, không cần đối chiếu tay.

12 file trong `notebooks/colab_steps/` được giữ lại làm tài liệu mô tả từng bước cho báo
cáo, nhưng phần tính toán thật đã chuyển hết vào `ml/preprocess/`.